# Генерация и сохранение chunks

Разбивает документы на chunks и сохраняет их в `data/03_chunks/`.

In [ ]:
import json
import re
from pathlib import Path

if Path('/workspace/data').exists():
    BASE = Path('/workspace')
else:
    BASE = Path.cwd().parent

NORM_DIR = BASE / 'data' / '02_normalized_text'
CHUNKS_DIR = BASE / 'data' / '03_chunks'
CHUNKS_DIR.mkdir(exist_ok=True)

norm_docs = sorted([d for d in NORM_DIR.iterdir() if d.is_dir()])
print(f'Документы: {len(norm_docs)}')

In [ ]:
HEADER_RE = re.compile(r'^(?:#+\s+)?(\d+(?:\.\d+)*)\s+([А-ЯЁ][А-Яа-яЁё \-,]{3,100})\.?$')

def build_breadcrumbs(text: str) -> list:
    lines = text.split('\n')
    sections = []
    stack = []
    for i, line in enumerate(lines):
        m = HEADER_RE.match(line.strip())
        if m:
            num = m.group(1)
            title = m.group(2).strip()
            level = num.count('.') + 1
            full = f'{num} {title}'
            while stack and stack[-1][0] >= level:
                stack.pop()
            stack.append((level, full))
            path = [h for _, h in stack]
            sections.append((i, full, path))
    return sections

def create_chunks(text: str, sections: list, chunk_size: int = 1500) -> list:
    lines = text.split('\n')
    chunks = []
    for i, (start_line, header, path) in enumerate(sections):
        if i + 1 < len(sections):
            end_line = sections[i + 1][0]
        else:
            end_line = len(lines)
        section_text = '\n'.join(lines[start_line:end_line]).strip()
        breadcrumb = ' > '.join(path)
        if len(section_text) > chunk_size:
            for j in range(0, len(section_text), chunk_size):
                chunk = section_text[j:j + chunk_size]
                chunks.append({
                    'section': breadcrumb,
                    'content': chunk,
                    'size': len(chunk),
                })
        else:
            chunks.append({
                'section': breadcrumb,
                'content': section_text,
                'size': len(section_text),
            })
    return chunks

In [ ]:
for doc_dir in norm_docs:
    norm_pages = []
    for p in sorted(doc_dir.glob('page_*.json')):
        try:
            d = json.loads(p.read_text(encoding='utf-8'))
            norm_pages.append(d)
        except Exception as e:
            print(f'Ошибка: {e}')
    
    full_text = '\n\n'.join((p.get('text') or '').strip() for p in norm_pages)
    sections = build_breadcrumbs(full_text)
    chunks = create_chunks(full_text, sections)
    
    doc_chunks_dir = CHUNKS_DIR / doc_dir.name
    doc_chunks_dir.mkdir(exist_ok=True)
    
    chunks_file = doc_chunks_dir / 'chunks.json'
    chunks_file.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding='utf-8')
    
    print(f'✅ {doc_dir.name}: {len(chunks)} chunks')